# Brent · `borough` (web: "Borough")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `borough` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

Note: the scope filter itself uses `borough` (Brent, or missing with `lpa_name` Brent).

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "borough"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "borough"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["borough"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,407,1.1
empty or blank,0,0.0
filled,36545,98.9
total,36952,100.0


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 407 | 1.10% |
| empty or blank | 0 | 0% |
| filled | 36,545 | 98.90% |

- **The 407 nulls are the 407 empty records** (known from the `status` notebook: the
  empty records are exactly Brent's records without a borough). They are in scope only
  through the "no borough, `lpa_name` Brent" part of the filter, and are dropped at load.
- So after cleaning, **every record has a `borough`**.

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["borough"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["borough"].value_counts().head(30).to_frame("count")

,count
borough,
str,36545
NoneType,407


values that are not strings: 0


,count
borough,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 36,545 |
| null | 407 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Most common values

Count identical values of `borough` in `df` (exact text). `shown_as` uses `repr()` so
hidden characters are visible.

In [5]:
print("distinct values:", df["borough"].nunique(), "| filled:", df["borough"].notna().sum())
top = df["borough"].value_counts(dropna=False).head(30).to_frame("count").reset_index()
top["shown_as"] = top["borough"].map(repr)
top

distinct values: 1 | filled: 36545


,borough,count,shown_as
0,Brent,36545,'Brent'
1,NaN,407,nan


### Findings · Step 3

Live API, 2026-11-10:

| `borough` | Count |
|---|---|
| `Brent` | 36,545 |
| null | 407 |

- **One value only:** every filled `borough` is exactly `Brent`. No case, spacing or
  other variants.
- This is partly by construction: the scope filter matches `borough.raw` = `Brent`
  exactly, so a variant such as `brent` or `Brent ` would not be in `df` at all. The one
  record that a full-text `match` on Brent finds but the exact `term` doesn't (open
  question 11) is outside this filter, so it can't show up here.